# Lab 8: Cryptography Fundamentals with Python
**Course:** FSCT 8561 -  Security Applications 

**Instructor:** Dr. Maryam R. Aliabadi

## Scenario
You have been hired as a Junior Security Engineer at CipherCorp. Your first task is to audit the company's legacy systems and migrate them to modern cryptographic standards before an upcoming security audit. 

## Learning Objectives
- Run and observe cryptographic algorithms in action.
- Understand hashing, symmetric encryption, key derivation, and secure encryption methods.
- Analyze why some cryptographic methods are secure or insecure.
- Perform a basic brute-force attack to understand the importance of salt and complexity.

## Part A — Hash Functions (One-Way functions)
**Goal:** Learn to generate cryptographic hashes and understand the concept of one-way encryption.
Hashing is used for integrity. If our database is stolen, we don't want a hacker to see "Password123". We want them to see a useless string.

### Task A1 — Basic Hashing 

In [11]:
import hashlib
password = input("Enter a password: ")
hash_object = hashlib.sha256(password.encode())
print("SHA-256 hash:", hash_object.hexdigest())

SHA-256 hash: 5d389f5e2e34c6b0bad96581c22cee0be36dcf627cd73af4d4cccacd9ef40cc3


**Observations:**
- Run twice with the same password → same hash?
- Change one character → observe hash change.

**Modification:**
- Change hashing algorithm: `hashlib.sha256` → `hashlib.md5` or `hashlib.sha1`.

**Questions:**
1. What happens when you slightly change the input?
2. Compare MD5 vs SHA-256 output length.
3. Why is hashing considered “one-way”?

### Task A2 — Hashing with Salt
Salt ensures that two users with the same password (e.g., "Spring2026") do not end up with the same hash in the database, protecting against Rainbow Table attacks.

In [12]:
import uuid

def hash_password(password):
    salt = uuid.uuid4().hex
    hashed = hashlib.sha256(salt.encode() + password.encode()).hexdigest()
    return hashed + ":" + salt

def check_password(stored_password, user_input):
    hashed, salt = stored_password.split(":")
    return hashed == hashlib.sha256(salt.encode() + user_input.encode()).hexdigest()

password = input("Enter password: ")
stored = hash_password(password)
print("Stored:", stored)
verify = input("Re-enter password: ")
print("Match:", check_password(stored, verify))

Stored: 799a09f12fe5314446b959a7254cfcbdf839467618c759f5122099cde908a689:231625f47c584c75b681f479308191e3
Match: True


**Questions:**
1. What is the role of salt?
2. Why does the hash change every time now?

---
## Part B — Symmetric Encryption with DES
**Goal:** Understand DES encryption, input requirements, and limitations.
DES is the "antique" of the crypto world. Its 56-bit key size is no longer secure against modern computing power. We use it here to understand the mechanics of block ciphers.

In [13]:
%pip install pycryptodome

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 15.7 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
from Crypto.Cipher import DES

key = "newkey12".encode("utf8")  # 8 bytes
message = "hello ".ljust(8).encode("utf8")  # 8 bytes

cipher = DES.new(key, DES.MODE_ECB)
cipher_text = cipher.encrypt(message)
print("Encrypted:", cipher_text)

cipher = DES.new(key, DES.MODE_ECB)
plain_text = cipher.decrypt(cipher_text)
print("Decrypted:", plain_text.decode())

Encrypted: b'V\xa1P8f\x10\xdc\xa1'
Decrypted: hello   


**Observations:**
- Run twice → compare outputs
- Change message slightly

**Modification:**
- Remove padding → observe error
- Fix padding with `.ljust(8)`
- Change key → observe output

**Questions:**
1. Why must DES input be exactly 8 bytes?
2. What happens if same key + same message is reused?
3. Why is this a potential security issue?

---
## Part C — AES Encryption
**Goal:** Understand AES encryption, key size, IV usage, and compare to DES.
AES is the industry standard used by governments. Unlike ECB mode used above, we use CBC (Cipher Block Chaining) with an Initialization Vector (IV) to ensure that the same message encrypted twice looks different every time.

In [40]:
from Crypto.Cipher import AES

# AES-128 requires a 16-byte key
key = "DifferentIV1234!".encode("utf8") 
iv = "This is an IV457".encode("utf8") 

cipher = AES.new(key, AES.MODE_CBC, iv)
# Data must be a multiple of 16 bytes
message = "Hello AES".ljust(16).encode("utf8")

cipher_text = cipher.encrypt(message)
print(f"AES Encrypted: {cipher_text.hex()}")

# Decrypt using the same IV
decipher = AES.new(key, AES.MODE_CBC, iv)
plain_text = decipher.decrypt(cipher_text)
print(f"Decrypted: {plain_text.decode().strip()}")

AES Encrypted: 24d4e789cc5a10c0e570266a2fe3f628
Decrypted: Hello AES


**Observations:**
- Run twice → same output?
- Change IV → observe effect

**Modification:**
- Use invalid key length → observe error
- Change message length → fix using `.ljust()`

**Questions:**
1. Why does AES require a specific key size?
2. What is the purpose of IV?
3. How is AES stronger than DES?

---
## Part D — Key Derivation (PBKDF2)
**Goal:** Learn secure key generation from passwords using PBKDF2.
Humans pick bad passwords. PBKDF2 makes it "expensive" for a hacker to guess passwords by running the hashing process thousands of times (iterations), slowing down brute-force attacks.

In [50]:
from Crypto.Protocol.KDF import PBKDF2
from Crypto import Random

password = "mypassword"
salt = b'fixedsalt1234567'

key = PBKDF2(password, salt, dkLen=16, count=100)
print("100 Iterations:", key)

key = PBKDF2(password, salt, dkLen=16, count=100000)
print("100000 Iterations:", key)

100 Iterations: b'\xd6\x17\xba\xd9\xb4\xb0d\xc7j\x16\xd7\xcb\xbd\x137G'
100000 Iterations: b"'[DXe\xd1\xce\x17z\xfdS0\xff\xfb\xc1\x7f"


C:\Users\suraj\AppData\Local\Temp\ipykernel_27176\4022910298.py:7: DeprecationWarning: PBKDF2 without 'hmac_hash_module' (HMAC-SHA1) uses weak defaults, which are deprecated and will be removed: pass them explicitly (for instance, count=600000, hmac_hash_module=SHA256)
  key = PBKDF2(password, salt, dkLen=16, count=100)
C:\Users\suraj\AppData\Local\Temp\ipykernel_27176\4022910298.py:10: DeprecationWarning: PBKDF2 without 'hmac_hash_module' (HMAC-SHA1) uses weak defaults, which are deprecated and will be removed: pass them explicitly (for instance, count=600000, hmac_hash_module=SHA256)
  key = PBKDF2(password, salt, dkLen=16, count=100000)


**Observations:**
- Run multiple times → compare outputs

**Modification:**
- Fix salt manually: `salt = b'fixedsalt123456'`
- Change iteration count → `count = 100` to `100000`

**Questions:**
1. Why does changing salt change the key?
2. What is the role of iterations?

---
## Part E — Fernet (Simplified Secure Encryption)
**Goal:** Encrypt/decrypt safely using Fernet and observe randomness.
In modern development, we often use Fernet because it handles the IV, padding, and integrity checks automatically, preventing "rookie" mistakes in implementation.

In [52]:
%pip install cryptography

   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   -------------------------------------- - 3.7/3.8 MB 22.8 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 18.8 MB/s  0:00:00

   ---------------------------------------- 0/3 [pycparser]
   ---------------------------------------- 0/3 [pycparser]
   ------------- -------------------------- 1/3 [cffi]
   ------------- -------------------------- 1/3 [cffi]
   ------------- -------------------------- 1/3 [cffi]
   ------------- -------------------------- 1/3 [cffi]
   -------------------------- ------------- 2/3 [cryptography]
   -------------------------- ------------- 2/3 [cryptography]
   -------------------------- ------------- 2/3 [cryptography]
   -------------------------- ------------- 2/3 [cryptography]
   -------------------------- ------------- 2/3 [cryptography]
   -------------------------- ------------- 2/3 [cryptography]
   -------------------------- ------------- 2/3 [cryptography

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [55]:
from cryptography.fernet import Fernet

key = Fernet.generate_key()
cipher = Fernet(key)

message = "Secret message".encode()
encrypted = cipher.encrypt(message)
print("Encrypted:", encrypted)

decrypted = cipher.decrypt(encrypted)
print("Decrypted:", decrypted.decode())

Encrypted: b'gAAAAABq-h0_mw1dv2E-J0NSnlGHfwAybk5YdRvIQTcw_KOWCIrORx-WwcMR_L8IOt2iceQybt2Rrq8opV-dG81qvZS9NGkwhw=='
Decrypted: Secret message


In [56]:

from cryptography.fernet import Fernet

key = Fernet.generate_key()
cipher = Fernet(key)

message = "Secret message".encode()
encrypted = cipher.encrypt(message)

print("Encrypted:", encrypted)

# Generate a different key
wrong_key = Fernet.generate_key()
wrong_cipher = Fernet(wrong_key)

# Attempt decryption using the wrong key
try:
    decrypted = wrong_cipher.decrypt(encrypted)
    print("Decrypted:", decrypted.decode())
except Exception as e:
    print("Decryption failed:", type(e).__name__)


Encrypted: b'gAAAAABq-h28xf7EvBw0CFY-Xpqf74Z3GxuiyYC_XkZXbpe6z371jDMmcH9JGmendyLcSID6Bw3aLQH999VzxUbgkTVdFl19WA=='
Decryption failed: InvalidToken


**Observations:**
- Run multiple times → compare outputs
- Modify key or save key to file → observe effect

**Questions:**
1. Why is Fernet safer than manual AES?
2. What happens if the key is incorrect?

---
## Part F — Broken Vault Challenge 
**Goal:** Observe weaknesses in deterministic encryption and weak password hashing.
A former employee was fired for poor security practices. They left behind an encrypted "vault" and a recovery PIN that was protected with a very weak hashing method. Your mission is to recover the PIN and identify why their encryption method was a failure.

### Task F1: Brute-Forcing the Admin PIN
The employee used MD5 to hash a 4-digit PIN (e.g., "0000" to "9999") without using a salt. You have recovered the target hash: 81dc9bdb52d04dc20036dbd8313ed055.

Instructions: Complete the Python script below to "brute-force" the PIN. You must iterate through every possible 4-digit combination until you find the one that matches the target hash.

In [57]:
import hashlib

target_hash = "81dc9bdb52d04dc20036dbd8313ed055"

print("Starting Brute Force Attack...")

# Loop through all possible 4-digit PINs
for i in range(10000):

    # Convert number to a 4-digit string
    pin = str(i).zfill(4)

    # Generate MD5 hash
    hashed_pin = hashlib.md5(pin.encode()).hexdigest()

    # Compare with target hash
    if hashed_pin == target_hash:
        print("PIN Found:", pin)
        print("Matching MD5 Hash:", hashed_pin)
        break

Starting Brute Force Attack...
PIN Found: 1234
Matching MD5 Hash: 81dc9bdb52d04dc20036dbd8313ed055


### Task F2: Analyzing the ECB Leak
The employee used DES in ECB mode to encrypt two different messages using the same key. Look at the output of the code below and answer the question that follows.

In [58]:
from Crypto.Cipher import DES

key = "security".encode("utf8")
cipher = DES.new(key, DES.MODE_ECB)

# Encrypting two separate data packets
packet_1 = "DATA_REQ".encode("utf8")
packet_2 = "DATA_REQ".encode("utf8")

print(f"Packet 1 Ciphertext: {cipher.encrypt(packet_1).hex()}")
print(f"Packet 2 Ciphertext: {cipher.encrypt(packet_2).hex()}")

Packet 1 Ciphertext: 6e9501ebbf08ec18
Packet 2 Ciphertext: 6e9501ebbf08ec18


**Question:**
If an attacker intercepts these two packets and sees that the encrypted "Ciphertext" is identical, what information have they gained even if they don't have the key? Why is this a major security vulnerability for a database containing many identical values (like "Male/Female" or "Yes/No")?

### Task F3 — Security Engineer's Recommendation (10 minutes)

Your team must propose a replacement design for the vault. Complete the table below in your report.

| Legacy design | Security weakness | Recommended replacement | Why? |
|---|---|---|---|
| Four-digit PIN | ? | ? | ? |
| Unsalted MD5 password verifier | ? | ? | ? |
| DES-ECB for sensitive data | ? | ? | ? |
| Hard-coded encryption key in source code | ? | ? | ? |

**Guidance:** Consider stronger authentication, unique salts with a suitable password KDF (e.g., Argon2id or properly configured PBKDF2), authenticated encryption (e.g., AES-GCM), and secure key storage. 

**Mini challenge:** A developer suggests `SHA256(PIN)` plus AES-CBC with a fixed IV. Identify **two remaining security problems**.


---
## Deliverables
A PDF containing:
- Screenshots of code outputs for each part (A–F)
- Your completed code for the Brute-Force loop.
- Answers to all questions and observations
- Submit as PDF: `Lab8-FirstName-LastName-StudentID.pdf`

## Good Luck!



## Grading Rubric — CipherCorp Security Audit

**Total: 100 Marks**

| Component | Assessment Criteria | Marks |
|---|---|---:|
| **Part A — Hashing and Salt** | A1: SHA-256 implementation, MD5 comparison, and observations (7). A2: Salted hashing, password verification, and explanation of salt (8). | **15** |
| **Part B — DES Encryption** | Successful encryption/decryption (4). Input-length and key modifications (3). Explanation of DES weaknesses and repeated ciphertext (3). | **10** |
| **Part C — AES Encryption** | Successful AES-CBC encryption/decryption (6). Key-length, IV, and message modifications (7). Explanation of IV purpose and AES versus DES (7). | **20** |
| **Part D — PBKDF2** | Successful key derivation (5). Salt and iteration experiments (5). Explanation of password-guessing resistance (5). | **15** |
| **Part E — Fernet Encryption** | Successful encryption/decryption (8). Key changes and repeat-encryption experiments (7). Explanation of authenticated encryption, key handling, and incorrect-key behavior (10). | **25** |
| **Part F — Broken Vault Challenge** | F1: PIN recovery and reasoning (5). F2: ECB experiment and interpretation (5). F3: Security recommendations and reflection (5). | **15** |
| **TOTAL** | | **100** |

---
